> Notebook-friendly copy of `part-I/1.4-matplotlib-and-xarray-solutions.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"cartopy": "cartopy", "pooch": "pooch", "xarray": "xarray"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# Solutions

**ℹ️ Reference solutions**

Worked solutions for [1.4-matplotlib-and-xarray-exercises.ipynb](1.4-matplotlib-and-xarray-exercises.ipynb). Run the shared setup cell below first — solutions 1 to 11 rely on its station records and random-number generator, as the exercises do. Every exercise has a worked solution, the four replication capstones included.

In [ ]:
# Pre-supplied: run this first. Three synthetic station records and a humidity series,
# standing in for a year of daily weather observations.
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(1)

In [ ]:
days = np.arange(366)                                  # day of year, 2024 (a leap year)
seasonal_celsius = -np.cos(2 * np.pi * days / 366) * 10.0

In [ ]:
valley_celsius = 6.0 + seasonal_celsius + rng.normal(0, 1.5, size=366)
ridge_celsius = valley_celsius - 6.5                   # ~1000 m higher
mountain_celsius = valley_celsius - 18.0               # ~2800 m higher

In [ ]:
# humidity peaks about a month after the coldest day, so it lags temperature
rh_percent = 68.0 + 12.0 * np.cos(2 * np.pi * (days - 30) / 366) + rng.normal(0, 3.0, size=366)

In [ ]:
print(valley_celsius.shape, rh_percent.shape)

## Exercise 1: Placing axes by hand

Build a 7 × 3 inch figure with two axes positioned with `fig.add_axes`: a main panel covering
roughly the left two-thirds, and a smaller one in the upper right. Plot the valley station's
whole year in the main panel and January alone (`days[:31]`) in the small one. Label the axes of
both, and give the small panel a title.

In [ ]:
fig = plt.figure(figsize=(7, 3))
main = fig.add_axes([0.08, 0.15, 0.58, 0.78])     # [left, bottom, width, height]
inset = fig.add_axes([0.76, 0.55, 0.22, 0.35])

main.plot(days, valley_celsius, color="tab:blue", linewidth=1)
main.set_xlabel("day of year")
main.set_ylabel("temperature (°C)")

inset.plot(days[:31], valley_celsius[:31], color="tab:blue")
inset.set_xlabel("day of year", fontsize=8)
inset.set_ylabel("(°C)", fontsize=8)
inset.set_title("January", fontsize=9)
plt.show()

## Exercise 2: A grid of panels

With `plt.subplots(2, 2, figsize=(9, 5))`, give each of the three station records and the
humidity series its own panel. Print the shape of the axes array, label every panel with its
quantity and unit, give each a title, and pack the figure with `tight_layout`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 5))
print(axes.shape)

axes[0, 0].plot(days, valley_celsius, color="tab:blue")
axes[0, 0].set_title("valley station")
axes[0, 1].plot(days, ridge_celsius, color="tab:orange")
axes[0, 1].set_title("ridge station")
axes[1, 0].plot(days, mountain_celsius, color="tab:green")
axes[1, 0].set_title("mountain station")

for ax in (axes[0, 0], axes[0, 1], axes[1, 0]):
    ax.set_xlabel("day of year")
    ax.set_ylabel("temperature (°C)")

axes[1, 1].plot(days, rh_percent, color="tab:purple")
axes[1, 1].set_title("relative humidity")
axes[1, 1].set_xlabel("day of year")
axes[1, 1].set_ylabel("relative humidity (%)")

plt.tight_layout()
plt.show()

## Exercise 3: Telling series apart without colour

Plot all three station records on one axes so that they stay distinguishable printed in black
and white: all three in black, one solid, one dashed, one dotted. Then add every thirtieth point
of the valley record as circular markers with no line joining them. Add a legend.

In a second figure, draw the valley record as it would read at twelve elevations, from 0 m to
2200 m every 200 m, cooling by 6.5 °C per 1000 m, and name each curve in a legend. The default
colour cycle has ten entries, so two of the twelve curves repeat earlier colours. Say in a comment
which ones, and make them distinguishable by drawing every curve after the tenth dashed.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(days, valley_celsius, color="black", linestyle="-", label="valley")
ax.plot(days, ridge_celsius, color="black", linestyle="--", label="ridge")
ax.plot(days, mountain_celsius, color="black", linestyle=":", label="mountain")
ax.plot(days[::30], valley_celsius[::30], color="black", linestyle="none",
        marker="o", markersize=5, label="valley, monthly sample")
ax.set_xlabel("day of year")
ax.set_ylabel("temperature (°C)")
ax.legend(fontsize=9)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for i, elevation_m in enumerate(np.arange(0, 2400, 200)):
    if i < 10:
        style = "-"
    else:
        style = "--"
    ax.plot(days, valley_celsius - 6.5 * elevation_m / 1000.0, linestyle=style,
            label=f"{elevation_m} m")
ax.set_xlabel("day of year")
ax.set_ylabel("temperature (°C)")
ax.legend(ncols=2, fontsize=8, title="elevation")
plt.show()

# with solid lines only, 2000 m and 2200 m (the 11th and 12th curves) reuse the colours of
# 0 m and 200 m; the dashes tell each pair apart

## Exercise 4: Ticks, gridlines, and axis limits

Plot the valley record with a tick at the first day of every month, labelled with the month's
initial (`"J"`, `"F"`, `"M"`, ...). The month lengths of 2024 are below; each month starts on the
running total of the lengths of the months before it, and an array's `.cumsum()` gives running
totals (1.3). Add major y ticks every 5 °C and minor ones every 1 °C, with a solid major grid and
a dotted minor grid.

In a second figure, show July and August only, taking the x-limits from the start days you
computed, and set the y-limits to leave 1 °C of space above and below the data in those two
months.

```python
month_days = np.array([31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])   # 2024
```

In [ ]:
month_days = np.array([31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])   # 2024
month_start = month_days.cumsum() - month_days     # running total of the months before
print(month_start)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(days, valley_celsius, color="tab:blue")
ax.set_xticks(month_start)
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_yticks(np.arange(-10, 21, 5))
ax.set_yticks(np.arange(-10, 21, 1), minor=True)
ax.grid(which="major", linewidth=1)
ax.grid(which="minor", linestyle=":", linewidth=0.4)
ax.set_xlabel("month, 2024")
ax.set_ylabel("temperature (°C)")
plt.show()

summer_celsius = valley_celsius[month_start[6]:month_start[8]]   # 1 July to 31 August

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(days, valley_celsius, color="tab:blue")
ax.set_xlim(month_start[6], month_start[8])
ax.set_ylim(summer_celsius.min() - 1, summer_celsius.max() + 1)
ax.set_xlabel("day of year")
ax.set_ylabel("temperature (°C)")
ax.set_title("July and August")
plt.show()

## Exercise 5: Annotate the largest jump

Find the largest one-day warming in the valley record: the day on which the temperature rose most
from the day before. The change from each day to the next is one slice of the record minus
another, offset by one day, and `np.argmax` of that change locates the jump. Print the day of year
of the warmer day and the size of the jump, mark the warmer day on the plot with `ax.annotate` and
an arrow, and add a plain `ax.text` label in an empty part of the figure.

In [ ]:
change_celsius = valley_celsius[1:] - valley_celsius[:-1]   # element i: day i to day i + 1
jump_day = int(np.argmax(change_celsius)) + 1               # the warmer of the two days
print("day", jump_day, "is", change_celsius[jump_day - 1], "°C warmer than the day before")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(days, valley_celsius, color="tab:blue")
ax.annotate("largest one-day warming", xy=(jump_day, valley_celsius[jump_day]),
            xytext=(150, -4), arrowprops={"facecolor": "k", "width": 1, "headwidth": 6})
ax.text(10, 15, "valley station, 2024", fontsize=9)
ax.set_xlabel("day of year")
ax.set_ylabel("temperature (°C)")
plt.show()

## Exercise 6: A parametric plot

The setup cell's humidity series lags temperature by about 30 days. Build a second one that lags
by 90 days (below), average the valley record and both humidity series into twelve monthly means
with `[:360].reshape(12, 30).mean(axis=1)`, and draw both loops on one axes: humidity against
temperature, a marker at each month, and a legend saying which lag is which. In a comment, say how
the loop changes as the lag grows, and what the twelve points would look like with no lag at all.

```python
rh_lag90_percent = 68.0 + 12.0 * np.cos(2 * np.pi * (days - 90) / 366) + rng.normal(0, 3.0, size=366)
```

In [ ]:
rh_lag90_percent = 68.0 + 12.0 * np.cos(2 * np.pi * (days - 90) / 366) + rng.normal(0, 3.0, size=366)

monthly_celsius = valley_celsius[:360].reshape(12, 30).mean(axis=1)
monthly_rh = rh_percent[:360].reshape(12, 30).mean(axis=1)
monthly_rh_lag90 = rh_lag90_percent[:360].reshape(12, 30).mean(axis=1)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(monthly_celsius, monthly_rh, marker="o", label="30-day lag")
ax.plot(monthly_celsius, monthly_rh_lag90, marker="s", linestyle="--", label="90-day lag")
ax.set_xlabel("monthly mean temperature (°C)")
ax.set_ylabel("monthly mean relative humidity (%)")
ax.set_title("humidity against temperature, month by month")
ax.legend()
plt.show()

# the longer the lag, the wider the loop: at 30 days the months trace a narrow loop along a
# falling diagonal, at 90 days -- a quarter of the annual cycle -- it opens out into a broad
# ellipse. With no lag, humidity would be a fixed function of temperature and the twelve
# points would fall on a single falling line

## Exercise 7: Comparing two distributions

Draw the distributions of daily temperature at the valley and mountain stations as two histograms
on one axes. Give them the same bin edges — `bins` also accepts an array of edges, which
`np.linspace` can build to span both records — and draw them semi-transparent with `alpha=0.6`, so
the overlap stays visible. Add a legend and label both axes. In a comment, say how the two
distributions differ, and why.

In [ ]:
edges_celsius = np.linspace(mountain_celsius.min(), valley_celsius.max(), 31)   # 30 shared bins

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(valley_celsius, bins=edges_celsius, alpha=0.6, label="valley")
ax.hist(mountain_celsius, bins=edges_celsius, alpha=0.6, label="mountain")
ax.set_xlabel("daily mean temperature (°C)")
ax.set_ylabel("count (days)")
ax.set_title("daily mean temperature, 2024")
ax.legend()
plt.show()

# the two histograms have the same shape, 18 °C apart, up to where the bin edges happen to fall:
# the mountain record is the valley record minus a constant, which moves a distribution without
# changing its spread

## Exercise 8: Monthly means as bars

Average the valley record into twelve monthly means with `[:360].reshape(12, 30).mean(axis=1)`
and draw them as a bar chart, one bar per month, with the month's initial as each tick label.
Colour the freezing months (mean below 0 °C) with the hex code `"#2c7bb6"` and the others with the
grey level `"0.6"` — `color` also accepts a list with one colour per bar. On a second axes beside
it, compare the annual means of the three stations with `barh`. Label the value axis with its unit
in both.

In [ ]:
monthly_celsius = valley_celsius[:360].reshape(12, 30).mean(axis=1)

bar_colours = []
for value_celsius in monthly_celsius:
    if value_celsius < 0:
        bar_colours.append("#2c7bb6")    # freezing
    else:
        bar_colours.append("0.6")

stations = ["valley", "ridge", "mountain"]
annual_mean_celsius = [valley_celsius.mean(), ridge_celsius.mean(), mountain_celsius.mean()]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# the initials repeat (three J, two M, two A), so they cannot be the bar positions themselves:
# matplotlib would draw every "J" bar at the same spot, on top of one another. Use positions
# 0-11 for the bars and the initials as tick labels
axes[0].bar(np.arange(12), monthly_celsius, color=bar_colours)
axes[0].set_xticks(np.arange(12))
axes[0].set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
axes[0].set_xlabel("month")
axes[0].set_ylabel("monthly mean temperature (°C)")

axes[1].barh(stations, annual_mean_celsius, color="tab:orange")
axes[1].set_xlabel("annual mean temperature (°C)")

plt.tight_layout()
plt.show()

## Exercise 9: One field, four ways

Build the small gridded field below, then draw it four times in a 2 × 2 figure: with `imshow`
(row 0 is the southernmost latitude, so make it appear at the bottom), with `pcolormesh` using
the coordinates as cell centres, with `contourf` using 8 levels, and with `pcolormesh` and
`shading="flat"`. The last needs the cell *edges*: one more value than there are centres along
each axis, starting half a grid step before the first centre. It should come out identical to the
second panel. Label the `imshow` panel's axes in grid index and the other three in degrees, give
each panel a labelled colorbar, and save the figure as `_files/field.svg`.

```python
lon = np.linspace(6.0, 9.0, 6)
lat = np.linspace(46.0, 47.5, 4)      # ascending: south -> north
field_celsius = 6.0 + (46.0 - lat)[:, None] * 3.0 + rng.normal(0, 1.0, size=(4, 6))
```

In [ ]:
from pathlib import Path

lon = np.linspace(6.0, 9.0, 6)
lat = np.linspace(46.0, 47.5, 4)
field_celsius = 6.0 + (46.0 - lat)[:, None] * 3.0 + rng.normal(0, 1.0, size=(4, 6))

# cell edges: half a grid step outside the first and last centres, one more than the centres
lon_step = lon[1] - lon[0]
lat_step = lat[1] - lat[0]
lon_edges = np.linspace(lon[0] - lon_step / 2, lon[-1] + lon_step / 2, lon.size + 1)
lat_edges = np.linspace(lat[0] - lat_step / 2, lat[-1] + lat_step / 2, lat.size + 1)
print(lon_edges, lat_edges)

fig, axes = plt.subplots(2, 2, figsize=(11, 6))

im = axes[0, 0].imshow(field_celsius, origin="lower")
fig.colorbar(im, ax=axes[0, 0], label="temperature (°C)")
axes[0, 0].set_title("imshow(origin='lower')")
axes[0, 0].set_xlabel("longitude index")
axes[0, 0].set_ylabel("latitude index")

pcm = axes[0, 1].pcolormesh(lon, lat, field_celsius, edgecolors="k")
fig.colorbar(pcm, ax=axes[0, 1], label="temperature (°C)")
axes[0, 1].set_title("pcolormesh, cell centres")

cf = axes[1, 0].contourf(lon, lat, field_celsius, 8)
fig.colorbar(cf, ax=axes[1, 0], label="temperature (°C)")
axes[1, 0].set_title("contourf, 8 levels")

pcf = axes[1, 1].pcolormesh(lon_edges, lat_edges, field_celsius, shading="flat", edgecolors="k")
fig.colorbar(pcf, ax=axes[1, 1], label="temperature (°C)")
axes[1, 1].set_title("pcolormesh, cell edges, shading='flat'")

for ax in (axes[0, 1], axes[1, 0], axes[1, 1]):
    ax.set_xlabel("longitude (°E)")
    ax.set_ylabel("latitude (°N)")

plt.tight_layout()
Path("_files").mkdir(exist_ok=True)
fig.savefig("_files/field.svg")
plt.show()

## Exercise 10: A vector field

Build the synthetic wind field below — a simple rotation, like a cyclone seen from above — and
draw it two ways in a 1 × 2 figure. On the left, `quiver` at every third grid point, over
`contour` lines of the wind speed. On the right, `streamplot` coloured by wind speed, with a
labelled colorbar whose unit, metres per second, is written with mathtext (1.4.5) so the
exponent is a real superscript. Label the axes in km.

```python
x_km = np.linspace(-100, 100, 25)
y_km = np.linspace(-100, 100, 25)
xx_km, yy_km = np.meshgrid(x_km, y_km)
u_ms = -yy_km / 8.0
v_ms = xx_km / 8.0
```

In [ ]:
x_km = np.linspace(-100, 100, 25)
y_km = np.linspace(-100, 100, 25)
xx_km, yy_km = np.meshgrid(x_km, y_km)
u_ms = -yy_km / 8.0
v_ms = xx_km / 8.0
speed_ms = np.sqrt(u_ms**2 + v_ms**2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

cs = axes[0].contour(xx_km, yy_km, speed_ms, colors="gray")
axes[0].clabel(cs, fmt="%2.0f")
axes[0].quiver(xx_km[::3, ::3], yy_km[::3, ::3], u_ms[::3, ::3], v_ms[::3, ::3])
axes[0].set_title("quiver over contours of wind speed (m s$^{-1}$)")

sp = axes[1].streamplot(x_km, y_km, u_ms, v_ms, density=1.2, color=speed_ms, cmap="viridis")
fig.colorbar(sp.lines, ax=axes[1], label="wind speed (m s$^{-1}$)")
axes[1].set_title("streamplot")

for ax in axes:
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")

plt.tight_layout()
plt.show()

## Exercise 11: Fix the map

The cell below draws a temperature anomaly field — the departure from a long-term mean, in °C, on
a 0.5° grid over the Alps — and breaks most of the checklist in 1.4.12. First list every practice
it breaks, as comments. Then redraw the same data so that it passes all of them.

In [ ]:
# Pre-supplied: a map that breaks most of the checklist.
# Leave this cell as it is -- write the corrected version in the next one.
anomaly_lon = np.linspace(5.0, 11.0, 13)
anomaly_lat = np.linspace(45.0, 48.0, 7)                     # ascending: south -> north
anomaly_celsius = ((46.5 - anomaly_lat)[:, None] * 1.5
                   + 0.4 * np.sin(anomaly_lon)[None, :]
                   + rng.normal(0, 0.3, size=(7, 13)))

fig, ax = plt.subplots(figsize=(3, 2))
ax.imshow(anomaly_celsius, cmap="jet")
ax.set_title("anomaly")
plt.show()

In [ ]:
# What the map above breaks:
# - imshow draws by array position: the axes are grid indices, not degrees, and with the
#   default origin='upper' the southernmost row is drawn at the top, so the map is upside down
# - there is no colorbar, so the colours carry no values and no unit
# - jet is a rainbow map: its bright bands draw boundaries the data do not have
# - the field is an anomaly that changes sign, but nothing in the colours marks zero
# - the title "anomaly" says nothing about what, where, or when
# - the figure is 3 x 2 inches, so the text is unreadable when projected

levels_celsius = np.linspace(-3.0, 3.0, 13)    # symmetric about zero, so white marks 0 °C

fig, ax = plt.subplots(figsize=(7, 4))
cf = ax.contourf(anomaly_lon, anomaly_lat, anomaly_celsius, levels_celsius,
                 cmap="RdBu_r", extend="both")
fig.colorbar(cf, ax=ax, label="temperature anomaly (°C)")
ax.set_xlabel("longitude (°E)", fontsize=12)
ax.set_ylabel("latitude (°N)", fontsize=12)
ax.set_title("2 m temperature anomaly over the Alps (synthetic)", fontsize=13)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

## Exercise 12: Build a labelled DataArray

From `data = np.arange(12.0).reshape(3, 4)`, build an xarray DataArray with dimensions
`("lat", "lon")`, latitude coordinates `[46.0, 46.5, 47.0]`, longitude coordinates
`[6.0, 6.5, 7.0, 7.5]`, and a `units` attribute of `"degC"`. Print its dims and its units.

In [ ]:
import xarray as xr

data = np.arange(12.0).reshape(3, 4)
da = xr.DataArray(
    data,
    dims=("lat", "lon"),
    coords={"lat": [46.0, 46.5, 47.0], "lon": [6.0, 6.5, 7.0, 7.5]},
    attrs={"units": "degC"},
)
print(da.dims)
print(da.attrs["units"])

## Exercise 13: Select, reduce, and subtract

Build the small Dataset below, then:

1. Print the spatial mean of the first day (by position) and the time mean at each longitude
   along 47.0° N (by label).
2. Select the grid point nearest 46.8° N, 6.3° E, and print its coordinates and its ten values.
3. Subtract each day's spatial mean — the mean over `lat` and `lon` — from the field. Print the
   dims of the result, and its spatial mean on the first day, which should be zero up to rounding.

```python
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2024-01-11", dtype="datetime64[D]")
lat = np.array([46.0, 46.5, 47.0]); lon = np.array([6.0, 6.5, 7.0, 7.5])
t = rng.normal(5, 3, size=(10, 3, 4))
ds = xr.Dataset({"t2m": (("time", "lat", "lon"), t)},
                coords={"time": time, "lat": lat, "lon": lon})
```

In [ ]:
import xarray as xr

In [ ]:
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2024-01-11", dtype="datetime64[D]")
lat = np.array([46.0, 46.5, 47.0]); lon = np.array([6.0, 6.5, 7.0, 7.5])
t = rng.normal(5, 3, size=(10, 3, 4))
ds = xr.Dataset({"t2m": (("time", "lat", "lon"), t)},
                coords={"time": time, "lat": lat, "lon": lon})

In [ ]:
print("first day spatial mean:", ds["t2m"].isel(time=0).mean().item())
print("time mean at lat=47.0:", ds["t2m"].sel(lat=47.0).mean(dim="time").values)

In [ ]:
point = ds["t2m"].sel(lat=46.8, lon=6.3, method="nearest")
print("nearest grid point:", point["lat"].item(), point["lon"].item())
print(point.values)

In [ ]:
departure = ds["t2m"] - ds["t2m"].mean(dim=("lat", "lon"))   # each day minus its own spatial mean
print("dims:", departure.dims)
print("spatial mean of the first day:", departure.isel(time=0).mean().item())

## Exercise 14: Monthly anomalies, two ways

Build the one-year daily temperature DataArray below and subtract its annual mean to get each
day's anomaly. Compute the monthly mean anomaly with `resample`, and print the calendar month
(1–12) whose mean anomaly is largest. Then compute the monthly mean anomaly again with
`groupby("time.month")`, and print the largest difference between the two sets of twelve values.
In a comment, say why the two agree here, and what would differ between them for a ten-year
record.

```python
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2025-01-01", dtype="datetime64[D]")
n = time.size; doy = np.arange(n)
t = 5 + -np.cos(2 * np.pi * doy / n) * 10 + rng.normal(0, 1.5, n)
da = xr.DataArray(t, dims="time", coords={"time": time}, attrs={"units": "degC"})
```

In [ ]:
import xarray as xr

In [ ]:
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2025-01-01", dtype="datetime64[D]")
n = time.size; doy = np.arange(n)
t = 5 + -np.cos(2 * np.pi * doy / n) * 10 + rng.normal(0, 1.5, n)
da = xr.DataArray(t, dims="time", coords={"time": time}, attrs={"units": "degC"})

In [ ]:
anomaly = da - da.mean(dim="time")

In [ ]:
monthly_anomaly = anomaly.resample(time="MS").mean()
print("monthly steps:", monthly_anomaly["time"].size)
print("largest mean anomaly in month", int(monthly_anomaly.argmax("time").item()) + 1)

In [ ]:
by_month = anomaly.groupby("time.month").mean()
print("largest difference:", np.abs(monthly_anomaly.values - by_month.values).max())

# for one year, each calendar month occurs once, so resampling and grouping average exactly the
# same days. Over ten years, resample would return 120 values, one per month of the record,
# while groupby("time.month") would still return 12, each averaging that calendar month over
# all ten years -- a monthly climatology

## Exercise 15: Round-trip through netCDF

Build a Dataset holding one 1D temperature variable, `t2m`, with a `units` attribute. Write it to
`_files/series.nc`, reopen it with `open_dataset`, and print the variable names and the recovered
units.

In [ ]:
import xarray as xr
from pathlib import Path

time = np.arange("2024-01-01", "2024-01-06", dtype="datetime64[D]")
ds = xr.Dataset({"t2m": ("time", np.arange(5.0))}, coords={"time": time})
ds["t2m"].attrs["units"] = "degC"

Path("_files").mkdir(exist_ok=True)
ds.to_netcdf("_files/series.nc")

reopened = xr.open_dataset("_files/series.nc")
print(list(reopened.data_vars))
print(reopened["t2m"].attrs["units"])
reopened.close()

## Exercise 16: What transform does

Rebuild the DataArray `da` from exercise 12 and draw it twice, side by side, on
`ccrs.EqualEarth()` maps cropped with `set_extent([5, 9, 45, 48], crs=ccrs.PlateCarree())` and
with `cfeature.BORDERS` added: once with `pcolormesh(..., transform=ccrs.PlateCarree())`, and
once with `transform` left out. Give the first panel a labelled colorbar. In a comment, say where
the field in the second panel went, and why no error was raised.

In [ ]:
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

data = np.arange(12.0).reshape(3, 4)
da = xr.DataArray(
    data,
    dims=("lat", "lon"),
    coords={"lat": [46.0, 46.5, 47.0], "lon": [6.0, 6.5, 7.0, 7.5]},
    attrs={"units": "degC"},
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), subplot_kw={"projection": ccrs.EqualEarth()})
for ax in axes:
    ax.set_extent([5, 9, 45, 48], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.BORDERS, linewidth=0.8)

pcm = axes[0].pcolormesh(da["lon"], da["lat"], da, transform=ccrs.PlateCarree())
fig.colorbar(pcm, ax=axes[0], label="temperature (°C)")
axes[0].set_title("with transform=ccrs.PlateCarree()")

axes[1].pcolormesh(da["lon"], da["lat"], da)      # no transform
axes[1].set_title("without transform")
plt.show()

# without transform, cartopy reads 6.0-7.5 and 46.0-47.0 as EqualEarth coordinates, which are
# metres from 0° N, 0° E: the field is drawn a few metres across in the Gulf of Guinea, far
# outside the cropped extent. Those are valid coordinates, so nothing raises an error

## Exercise 17: Choosing a projection

Each figure described below needs a map. For each, choose a projection from 1.4.13 and draw the
empty map — land, ocean, and coastline, cropped with `set_extent` where the figure covers less
than the globe. Draw the three as panels of one figure; each panel needs its own projection, so
create each axes with `fig.add_subplot`. Justify each choice in a comment of one or two sentences.

1. The extent of Arctic sea ice in September. `ccrs.NorthPolarStereo()` is the northern
   counterpart of `SouthPolarStereo`.
2. How much of each continent is covered by tropical forest, for a reader who will compare the
   areas by eye.
3. A warm anomaly spreading through the Southern Ocean into the Atlantic, Indian, and Pacific
   basins, for a figure whose point is that the Southern Ocean links all three.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(14, 5))

# 1. a polar stereographic map centred on the pole shows the whole Arctic ice cover with little
#    distortion, and is the standard projection for sea-ice products
ax = fig.add_subplot(1, 3, 1, projection=ccrs.NorthPolarStereo())
ax.set_extent([-180, 180, 55, 90], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.OCEAN, facecolor="#a6cee3")
ax.add_feature(cfeature.LAND, facecolor="#b2df8a")
ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
ax.set_title("Arctic sea ice: NorthPolarStereo")

# 2. comparing areas by eye needs an equal-area map, on which twice the forest covers twice the
#    paper; EqualEarth keeps every region at its true relative size
ax = fig.add_subplot(1, 3, 2, projection=ccrs.EqualEarth())
ax.add_feature(cfeature.OCEAN, facecolor="#a6cee3")
ax.add_feature(cfeature.LAND, facecolor="#b2df8a")
ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
ax.set_title("forest area by continent: EqualEarth")

# 3. Spilhaus is centred on Antarctica and cuts the continents rather than the ocean, so the
#    Southern Ocean appears as one ring joining the three basins; a standard world map stretches
#    it along the bottom edge and cuts it where the left and right edges meet
ax = fig.add_subplot(1, 3, 3, projection=ccrs.Spilhaus())
ax.add_feature(cfeature.OCEAN, facecolor="#a6cee3")
ax.add_feature(cfeature.LAND, facecolor="#b2df8a")
ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
ax.set_title("Southern Ocean link: Spilhaus")

plt.tight_layout()
plt.show()

## Exercise 18: NASA's real global-temperature record

The file cached below is NASA GISS's monthly global-mean surface temperature anomaly (°C,
relative to a 1951–1980 baseline), one row per year since 1880, one column per calendar month.
A handful of recent months are still missing, marked `***`.

Wrap the twelve month columns in an xarray `DataArray` with dims `("year", "month")` and a `units`
attribute, then draw two figures: the annual mean as a line against year, and the full
`(year, month)` array with xarray's own `.plot()`. In a comment, compare what the warming trend
looks like read month by month against the single annual line.

In [ ]:
import xarray as xr
import pooch

path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/nasa_gistemp_global_temp_anomaly.csv",
    known_hash="sha256:6cfa44e7bbacd9b12cb10bdd64b3182c2735fa3f3a95688e1f7bc8e5dfcece93",
    fname="nasa_gistemp_global_temp_anomaly.csv",
    path=pooch.os_cache("mlees"),
)
data = np.genfromtxt(path, skip_header=1, delimiter=",", names=True,
                     missing_values="***", filling_values=np.nan)

anomaly = np.column_stack((data["Jan"], data["Feb"], data["Mar"], data["Apr"],
                           data["May"], data["Jun"], data["Jul"], data["Aug"],
                           data["Sep"], data["Oct"], data["Nov"], data["Dec"]))

da = xr.DataArray(
    anomaly,
    dims=("year", "month"),
    coords={"year": data["Year"], "month": np.arange(1, 13)},
    attrs={"units": "degC", "long_name": "global-mean surface temperature anomaly"},
)

fig, ax = plt.subplots(figsize=(7, 3))
da.mean(dim="month").plot(ax=ax, color="tab:red")   # skips missing months automatically
ax.set_xlabel("year")
ax.set_ylabel("temperature anomaly (°C)")
ax.set_title("annual-mean global temperature anomaly")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
da.plot(ax=ax)          # colorbar labelled from long_name and units
ax.set_title("month-by-month anomaly")
plt.show()

# the annual line smooths out a within-year signal: the heatmap shows that recent decades
# are warm in every month, not just on average, and that the trend is not perfectly uniform
# across months

## Replicating plots

The four exercises below each show a figure built from a real dataset and ask you to rebuild it
as closely as you can. The data arrive through a pre-supplied `pooch` cell; everything after that
is yours.

Work towards the target one element at a time — get the data onto the axes first, then the
coordinates, then the colours, then the labels — rather than trying to write the whole cell in
one go.

## Exercise 19: Global surface air temperature and its zonal mean

The three files cached below hold one snapshot of surface air temperature from the
[NCEP/NCAR atmospheric reanalysis 1](https://psl.noaa.gov/data/gridded/data.ncep.reanalysis.html),
as plain numpy arrays: `lon` (192 longitudes), `lat` (94 latitudes, running north to south), and
`temp_kelvin`, the field itself with shape `(94, 192)` — **in kelvin**.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.4-target-global-temperature.png" alt="A filled contour map of global surface air temperature in magma colours with a white dashed contour at minus ten degrees, beside a line plot of the zonal mean against latitude" width="100%">

<em>The figure to replicate. Left: the field as filled contours, with a single white contour drawn
at −10 °C. Right: the zonal mean — the average along longitude — against latitude.</em>


Rebuild it, and say in a comment which hemisphere's winter this snapshot was taken in, and how
the zonal mean tells you.

In [ ]:
# Pre-supplied: download the three arrays and cache them locally.
import pooch

In [ ]:
base_url = "https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/"

In [ ]:
path_lon = pooch.retrieve(
    url=base_url + "ncep_global_temp_lon.npy",
    known_hash="sha256:eaf54b88dd89279d3034da17fe8470dc2c841bf9fa89b2aa741dacff9c326cdb",
    fname="ncep_global_temp_lon.npy",
    path=pooch.os_cache("mlees"),
)
path_lat = pooch.retrieve(
    url=base_url + "ncep_global_temp_lat.npy",
    known_hash="sha256:af1f438080460e1fca4583b2ec19b44285a3d3776e4d21b8da9b6e162906c88a",
    fname="ncep_global_temp_lat.npy",
    path=pooch.os_cache("mlees"),
)
path_temp = pooch.retrieve(
    url=base_url + "ncep_global_temp_kelvin.npy",
    known_hash="sha256:e040ca257334708b43e86398e09a5669fcf051179ecf5dcd278f758d67beed20",
    fname="ncep_global_temp_kelvin.npy",
    path=pooch.os_cache("mlees"),
)

In [ ]:
lon = np.load(path_lon)
lat = np.load(path_lat)
temp_celsius = np.load(path_temp) - 273.15      # the file is in kelvin
print(lon.shape, lat.shape, temp_celsius.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={"width_ratios": [5, 1.5]})
ax0, ax1 = axes

cf = ax0.contourf(lon, lat, temp_celsius, cmap="magma",
                  levels=np.linspace(-30, 40, 15), extend="both")
ax0.contour(lon, lat, temp_celsius, colors="w", levels=[-10], extend="both")
fig.colorbar(cf, ax=ax0, label="temperature (°C)")
ax0.set_xlabel("longitude (°E)")
ax0.set_ylabel("latitude (°N)")
ax0.set_title("global surface air temperature")

ax1.plot(np.nanmean(temp_celsius, axis=1), lat, linewidth=2, color="k")
ax1.set_xlabel("temperature (°C)")
ax1.set_ylabel("latitude (°N)")
ax1.set_title("zonal mean")
ax1.grid()

plt.tight_layout()
plt.show()

# northern-hemisphere winter: the zonal mean reaches about -39 °C at the northernmost latitude
# but only about -18 °C at the southernmost, and it stays near 26 °C out to 20° S while it has
# already dropped to about 22 °C by 20° N -- the warm belt is skewed towards the summer
# hemisphere, which here is the southern one

## Exercise 20: Historic significant earthquakes

The file cached below is NOAA's catalog of significant earthquakes, reaching back to 2150 BCE:
one row per event, tab-separated, 47 columns. The four you need are at positions 8
(`FOCAL_DEPTH`, km), 9 (`EQ_PRIMARY`, magnitude), 20 (`LATITUDE`) and 21 (`LONGITUDE`).

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.4-target-earthquakes-scatter.png" alt="A scatter plot of earthquake longitude against latitude, coloured by the base-10 logarithm of focal depth and sized by magnitude" width="100%">

<em>The figure to replicate: every event as one point, coloured by the base-10 logarithm of its
focal depth and sized by its magnitude. No coastline is drawn — the plate boundaries are the
data.</em>


Rebuild it, and in a comment name the feature of the Earth that the points trace out.

In [ ]:
import pooch

path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/signif.txt.tsv",
    known_hash="sha256:f27e00d335f64ffaf437e74a8efb54f9154d40b417ff93fefb0fd64856e6d345",
    fname="signif.txt.tsv",
    path=pooch.os_cache("mlees"),
)
arr = np.genfromtxt(path, delimiter="\t", skip_header=1)
focal_depth_km = arr[:, 8]
magnitude = arr[:, 9]
latitude = arr[:, 20]
longitude = arr[:, 21]

keep = (focal_depth_km > 0) & (magnitude > 0) & ~np.isnan(latitude) & ~np.isnan(longitude)
focal_depth_km, magnitude = focal_depth_km[keep], magnitude[keep]
latitude, longitude = latitude[keep], longitude[keep]
print("events kept:", keep.sum(), "of", keep.size)

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(longitude, latitude, s=magnitude ** 2, c=np.log10(focal_depth_km),
                marker="o", cmap="viridis", alpha=0.7)
fig.colorbar(sc, ax=ax, label="log10(focal depth in km)")
ax.set_xlabel("longitude (°E)")
ax.set_ylabel("latitude (°N)")
ax.set_title("historic significant earthquakes: depth and magnitude")
ax.grid(linestyle=":", linewidth=0.5)
plt.tight_layout()
plt.show()

# the points fall along plate boundaries, and the empty areas are plate interiors: the deepest
# events (yellow) sit where one plate is being subducted beneath another, along the Pacific rim
# and the Indonesian arc

## Exercise 21: Antarctic sea ice, summer against winter

Two real snapshots of Antarctic sea ice concentration from NOAA/NSIDC are cached below, as
`path_winter` (1 August 2017) and `path_summer` (31 December 2017). Each holds
`seaice_conc_cdr`, the ice concentration as a fraction from 0 to 1, on a south polar
stereographic grid — its coordinates are 2D `latitude`/`longitude` arrays rather than simple 1D
dimensions, so xarray's own `.plot()` will not label it cleanly.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.4-target-seaice.png" alt="Two south polar stereographic maps of Antarctic sea ice concentration, one in austral winter with a wide ice ring and one in austral summer with much less ice" width="100%">

<em>The figure to replicate: the same field on the same projection at two dates, with the land drawn
in grey beneath the ice and one colorbar shared by both panels.</em>


Rebuild it, and say which date shows more ice.

In [ ]:
# Pre-supplied: download the data files and cache them locally.
import pooch

path_winter = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/seaice_conc_daily_sh_f17_20170801_v03r01.nc",
    known_hash="sha256:1ff50bca1e6249a9b2fcd9d9466e31bdb5be650243f99c7319ab2ce625b87ce7",
    fname="seaice_conc_daily_sh_f17_20170801_v03r01.nc",
    path=pooch.os_cache("mlees"),
)
path_summer = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/seaice_conc_daily_sh_f17_20171231_v03r01.nc",
    known_hash="sha256:309418969ad09f42b8104589bcb86de4ed353a5742fef9385baec174c7d55e66",
    fname="seaice_conc_daily_sh_f17_20171231_v03r01.nc",
    path=pooch.os_cache("mlees"),
)

In [ ]:
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

paths = [path_winter, path_summer]
titles = ["1 August 2017 (austral winter)", "31 December 2017 (austral summer)"]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5),
                         subplot_kw={"projection": ccrs.SouthPolarStereo()})

mean_conc = []
for ax, path, title in zip(axes, paths, titles):
    ds = xr.open_dataset(path)
    conc = ds["seaice_conc_cdr"].isel(time=0)
    conc = conc.where(conc <= 1)          # drop the land/coast/missing flag values above 1
    mean_conc.append(float(conc.mean()))

    ax.set_extent([-180, 180, -90, -55], ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, edgecolor="k", facecolor="0.8")
    ax.gridlines(linewidth=0.3, color="0.5")
    pcm = ax.pcolormesh(ds["longitude"], ds["latitude"], conc,
                        transform=ccrs.PlateCarree(), vmin=0, vmax=1, cmap="Blues_r")
    ax.set_title(title)
    ds.close()

fig.colorbar(pcm, ax=axes, label="sea ice concentration (fraction)")
plt.show()

print(f"mean concentration, winter: {mean_conc[0]:.3f} | summer: {mean_conc[1]:.3f}")

# August, the austral winter, shows far more ice: a broad ring reaching well away from the
# coast, against a thin and broken fringe at the end of December

## Exercise 22: The 2014 earthquakes over North America

The file cached below is the real USGS earthquake catalog for 2014 — one row per event
worldwide, with latitude, longitude, depth and magnitude in columns 1 to 4. The target figure
keeps the whole catalog and lets the map extent do the selecting, so what you see is a window
onto a global dataset rather than one cropped to North America in advance.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.4-target-earthquakes-map.png" alt="A Robinson-projection map of North America with state boundaries, lakes and rivers, and earthquake locations coloured by depth" width="90%">

<em>The figure to replicate: magnitude 4 and above, on a Robinson projection cropped to North and
Central America, over land, ocean, lake, river and state-boundary features.</em>


Rebuild it.

In [ ]:
# Pre-supplied: download the data file and cache it locally.
import pooch

path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/usgs_earthquakes_2014.csv",
    known_hash="sha256:84d455fb96dc8f782fba4b5fbe56cb8970cab678f07c766fcba1b1c4674de1b1",
    fname="usgs_earthquakes_2014.csv",
    path=pooch.os_cache("mlees"),
)

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

arr = np.genfromtxt(path, delimiter=",", skip_header=1, usecols=(1, 2, 3, 4))
latitude, longitude, depth_km, mag = arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3]

keep = (mag >= 4) & (depth_km > 0)
latitude, longitude = latitude[keep], longitude[keep]
depth_km, mag = depth_km[keep], mag[keep]
print("events kept:", keep.sum(), "of", keep.size)

fig, ax = plt.subplots(figsize=(9, 7), subplot_kw={"projection": ccrs.Robinson()})
ax.set_extent([-140, -60, 12, 70], ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, edgecolor="k", linewidth=1.0, facecolor="0.92")
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.LAKES)
ax.add_feature(cfeature.RIVERS)
ax.add_feature(cfeature.STATES, edgecolor="gray", linewidth=0.6)
ax.gridlines(linewidth=0.3, color="0.5")

sc = ax.scatter(longitude, latitude, transform=ccrs.PlateCarree(),
                c=np.log10(depth_km), s=mag**2, cmap="viridis", alpha=0.7)
fig.colorbar(sc, ax=ax, label="log10(depth in km)")
ax.set_title("2014 earthquakes, magnitude 4 and above")
plt.show()

# the map window is what selects: the catalog is worldwide, and what shows up here is the
# Aleutian arc, the Californian plate boundary, the Central American trench, and a tight
# cluster in Oklahoma that is induced rather than tectonic